# ESD RAG Support Assistant v3

RAG system for SNOW knowledge articles.

**Accepts:** `.mhtml` (download from SNOW) | `.html` | `.txt` (raw or Codex-enriched)

**Outputs:** Troubleshooting steps | SNOW work notes | Slack messages | Escalation paths

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu beautifulsoup4 lxml

## Step 2: Upload Articles

Upload your files — supports all three formats:
- `.mhtml` — direct download from SNOW (recommended)
- `.html` — alternative download format
- `.txt` — raw text or Codex-enriched articles

In [ ]:
import os
from google.colab import files

os.makedirs('articles', exist_ok=True)

print("Select your article files (.mhtml, .html, or .txt)...")
uploaded = files.upload()

for filename, content in uploaded.items():
    filepath = os.path.join('articles', filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    ext = os.path.splitext(filename)[1]
    print(f"  Saved: {filename} ({ext})")

all_files = [f for f in os.listdir('articles') if f.endswith(('.txt', '.html', '.mhtml', '.mht'))]
print(f"\nTotal articles: {len(all_files)}")

## Step 3: Build RAG Index

Parses all articles (auto-detects format), creates embeddings, builds FAISS index.

Re-run after adding new articles.

In [ ]:
import re
import email
import quopri
import base64
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss


def extract_text_from_mhtml(filepath):
    """Extract clean text from an MHTML file."""
    with open(filepath, 'rb') as f:
        raw = f.read()

    # Try parsing as MIME message
    try:
        msg = email.message_from_bytes(raw)
        html_content = ''

        if msg.is_multipart():
            for part in msg.walk():
                content_type = part.get_content_type()
                if content_type == 'text/html':
                    payload = part.get_payload(decode=True)
                    if payload:
                        charset = part.get_content_charset() or 'utf-8'
                        try:
                            html_content = payload.decode(charset, errors='ignore')
                        except:
                            html_content = payload.decode('utf-8', errors='ignore')
                    break
        else:
            payload = msg.get_payload(decode=True)
            if payload:
                charset = msg.get_content_charset() or 'utf-8'
                try:
                    html_content = payload.decode(charset, errors='ignore')
                except:
                    html_content = payload.decode('utf-8', errors='ignore')

        if html_content:
            return extract_text_from_html_string(html_content)
    except:
        pass

    # Fallback: read as text and find HTML portion
    try:
        text = raw.decode('utf-8', errors='ignore')
        html_match = re.search(r'<html[^>]*>.*?</html>', text, re.DOTALL | re.IGNORECASE)
        if html_match:
            return extract_text_from_html_string(html_match.group())
        # Last fallback: just strip anything that looks like MIME headers
        lines = text.split('\n')
        content_lines = []
        past_headers = False
        for line in lines:
            if past_headers:
                content_lines.append(line)
            elif line.strip() == '':
                past_headers = True
        return '\n'.join(content_lines)
    except:
        return raw.decode('utf-8', errors='ignore')


def extract_text_from_html_string(html_string):
    """Extract clean text from HTML string using BeautifulSoup."""
    soup = BeautifulSoup(html_string, 'lxml')

    # Remove script and style elements
    for element in soup(['script', 'style', 'meta', 'link', 'head']):
        element.decompose()

    # Get text
    text = soup.get_text(separator='\n')

    # Clean up: remove excessive blank lines and whitespace
    lines = [line.strip() for line in text.split('\n')]
    lines = [line for line in lines if line]  # remove empty lines
    text = '\n'.join(lines)

    # Remove very short lines that are likely UI artifacts
    # (buttons, icons, navigation items)
    filtered = []
    for line in text.split('\n'):
        # Keep lines that have substance
        if len(line) > 2:
            filtered.append(line)
    return '\n'.join(filtered)


def extract_text_from_html_file(filepath):
    """Extract text from an HTML file."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return extract_text_from_html_string(f.read())


def parse_article(filepath):
    """Parse any supported article format into structured data."""
    filename = os.path.basename(filepath)
    ext = os.path.splitext(filename)[1].lower()

    # Extract text based on file type
    if ext in ('.mhtml', '.mht'):
        content = extract_text_from_mhtml(filepath)
        source_type = 'mhtml'
    elif ext == '.html':
        content = extract_text_from_html_file(filepath)
        source_type = 'html'
    else:  # .txt
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        source_type = 'txt'

    article = {
        'filepath': filepath,
        'filename': filename,
        'source_type': source_type,
        'full_text': content,
        'title': filename.replace('.mhtml', '').replace('.mht', '').replace('.html', '').replace('.txt', '').replace('_', ' ').replace('-', ' ').title(),
        'system': '',
        'category': '',
        'keywords': '',
        'synonyms': '',
        'escalation_team': '',
        'troubleshooting': '',
        'resolution': '',
        'escalation_info': '',
        'symptoms': ''
    }

    # Try to extract Codex-enriched tags (works for .txt enriched files)
    tag_patterns = {
        'title': r'\[TITLE\]\s*(.+)',
        'system': r'\[SYSTEM\]\s*(.+)',
        'category': r'\[CATEGORY\]\s*(.+)',
        'keywords': r'\[KEYWORDS\]\s*(.+)',
        'synonyms': r'\[SYNONYMS\]\s*(.+)',
        'escalation_team': r'\[ESCALATION_TEAM\]\s*(.+)',
        'symptoms': r'\[SYMPTOMS\]\s*(.+)',
    }

    for field, pattern in tag_patterns.items():
        match = re.search(pattern, content, re.IGNORECASE)
        if match:
            article[field] = match.group(1).strip()

    # Extract multi-line sections
    section_patterns = {
        'troubleshooting': r'\[TROUBLESHOOTING\]\s*\n(.*?)(?=\n\[|$)',
        'resolution': r'\[RESOLUTION\]\s*\n(.*?)(?=\n\[|$)',
        'escalation_info': r'\[ESCALATION\]\s*\n(.*?)(?=\n\[|$)',
    }

    for field, pattern in section_patterns.items():
        match = re.search(pattern, content, re.IGNORECASE | re.DOTALL)
        if match:
            article[field] = match.group(1).strip()

    # Auto-detect system from filename or content
    if not article['system']:
        systems = ['Zeus', 'Five9', 'Okta', 'OA', 'VPN', 'Zoom', 'Zscaler',
                   'Payments', 'Hardware', 'Citrix', 'SAP', 'Kronos', 'ADP',
                   'GCP', 'Chrome', 'Windows', 'Slack', 'Teams']
        text_lower = (filename + ' ' + content[:1000]).lower()
        for sys in systems:
            if sys.lower() in text_lower:
                article['system'] = sys
                break

    return article


# --- Load all articles ---
articles = []
for fname in sorted(os.listdir('articles')):
    if fname.endswith(('.txt', '.html', '.mhtml', '.mht')):
        filepath = os.path.join('articles', fname)
        article = parse_article(filepath)
        articles.append(article)

        # Status indicator
        if article['keywords']:
            status = 'enriched'
        else:
            status = article['source_type']
        sys_tag = article['system'] or '?'
        text_len = len(article['full_text'])
        print(f"  [{status}] [{sys_tag}] {article['title']} ({text_len} chars)")

if not articles:
    print("No articles found. Run Step 2 to upload files.")
else:
    # Build search corpus
    search_texts = []
    for art in articles:
        parts = [
            art['title'],
            art['system'],
            art['category'],
            art['keywords'],
            art['synonyms'],
            art['symptoms'],
            art['troubleshooting'],
            art['resolution'],
            # For non-enriched articles, use more of the full text
            art['full_text'][:2000] if not art['keywords'] else ''
        ]
        search_texts.append(' | '.join([p for p in parts if p]))

    print("\nLoading embedding model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')

    print("Creating embeddings...")
    embeddings = model.encode(search_texts, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    print(f"\nIndex ready — {index.ntotal} articles indexed")
    print("Go to Step 4 to query!")

## Step 4: Query

Describe the issue and get all outputs. Run as many times as you want.

In [ ]:
def search_articles(query, top_k=3):
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, min(top_k, len(articles)))
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(articles):
            results.append({
                'article': articles[idx],
                'distance': distances[0][i],
                'relevance': max(0, 1 - distances[0][i] / 2)
            })
    return results


def get_steps(article):
    if article['troubleshooting']:
        return article['troubleshooting']
    lines = article['full_text'].split('\n')
    steps = [l.strip() for l in lines if re.match(r'^\d+[.\)]', l.strip())]
    if steps:
        return '\n'.join(steps)
    return article['full_text'][:500] + '...'


def generate_work_notes(query, article, caller):
    steps = get_steps(article)
    resolution = article['resolution'] or 'See troubleshooting steps above.'
    return f"""--- Work Notes ---\nContacted {caller} regarding: {query}\nKB Reference: {article['title']}\nSystem: {article['system'] or 'N/A'}\n\nTroubleshooting performed:\n{steps}\n\nResolution: {resolution}\n--- End Work Notes ---"""


def generate_slack_message(query, article, caller):
    steps = get_steps(article)
    step_lines = [l.strip() for l in steps.split('\n') if re.match(r'^\d+[.\)]', l.strip())]
    first_steps = '\n'.join(step_lines[:3]) if step_lines else 'I will review your case shortly.'
    system_name = article['system'] or 'your application'
    return f"""Hello {caller}! :pika-wave:\n\nMy name is Lalo from IT Support, and I'm reaching out regarding your recent ticket.\n\nI see you're experiencing an issue with *{system_name}*. Let's try a few things:\n\n{first_steps}\n\nCould you try these and let me know the result? If the issue persists, I'll take a closer look.\n\nYou can also reach me on Zoom: https://oportun.zoom.us/my/lalo.esd\n\nThank you! :star2:"""


def generate_escalation(article):
    esc_team = article['escalation_team'] or 'Not specified'
    esc_info = article['escalation_info'] or 'Include: username, steps taken, error messages, screenshots.'
    return f"""ESCALATION\nSystem: {article['system'] or 'N/A'}\nEscalate to: {esc_team}\n\n{esc_info}"""


# === Interactive Query ===
print("=" * 60)
print("ESD RAG Support Assistant v3")
print("=" * 60)

query = input("\nIssue description (or paste Short Description):\n> ")

if query.strip():
    caller = input("Caller name (Enter to skip): ").strip() or "[User Name]"

    results = search_articles(query)

    if results:
        best = results[0]
        art = best['article']
        pct = f"{best['relevance']*100:.0f}%"

        print(f"\n{'=' * 60}")
        print(f"MATCH: {art['title']}")
        print(f"   Relevance: {pct} | System: {art['system'] or '?'} | Source: {art['source_type']}")
        if art['keywords']:
            print(f"   Keywords: {art['keywords']}")
        print(f"{'=' * 60}")

        print(f"\n{'_' * 50}")
        print("TROUBLESHOOTING STEPS")
        print(f"{'_' * 50}")
        print(get_steps(art))

        print(f"\n{'_' * 50}")
        print("SNOW WORK NOTES — copy and paste")
        print(f"{'_' * 50}")
        print(generate_work_notes(query, art, caller))

        print(f"\n{'_' * 50}")
        print("SLACK MESSAGE — copy and paste")
        print(f"{'_' * 50}")
        print(generate_slack_message(query, art, caller))

        print(f"\n{'_' * 50}")
        print("ESCALATION PATH")
        print(f"{'_' * 50}")
        print(generate_escalation(art))

        if len(results) > 1:
            print(f"\n{'_' * 50}")
            print("OTHER MATCHES")
            print(f"{'_' * 50}")
            for r in results[1:]:
                a = r['article']
                rel = f"{r['relevance']*100:.0f}%"
                print(f"  [{rel}] {a['title']} ({a['system'] or '?'})")
    else:
        print("\nNo matches found.")
else:
    print("No query entered.")

## Step 5 (Optional): Preview extracted text

Use this to check what text was extracted from your MHTML/HTML files.

In [ ]:
# Show extracted text from all articles
for i, art in enumerate(articles):
    print(f"\n{'=' * 60}")
    print(f"[{i+1}] {art['title']} ({art['source_type']})")
    print(f"System: {art['system'] or 'not detected'}")
    print(f"Text length: {len(art['full_text'])} chars")
    print(f"{'=' * 60}")
    # Show first 500 chars
    print(art['full_text'][:500])
    if len(art['full_text']) > 500:
        print(f"\n... ({len(art['full_text']) - 500} more chars)")
    print()

---
### Tips
- Download KBs from SNOW as .mhtml and upload directly
- Use Step 5 to verify the text was extracted correctly
- For better matching: run articles through Codex to add keywords/synonyms, save as .txt, upload those
- Mix formats freely — you can upload .mhtml, .html, and .txt together
- Re-run Step 3 after adding new articles
- File > Save a copy in Drive